# Model development

In [1]:
from sanity_functions import *

In [2]:
transactions = pd.read_csv('raw_transactions.csv')
transactions.head()

,event_created_at,amount,infraction,merchant_id,mc_weeks_signup,mc_atm_br_tx_out_count_lifetime,mc_credit_transfer_out_br_sum_last_30d,mc_billpay_br_sum_last_15d,mc_billpay_br_count_last_1y,mc_credit_transfer_out_br_sum_last_1y,...,mc_tx_cp_dbt_sum_sq_fail_last_14d,mc_merchants_ip_count_dist_shared_last_15d,mcc_sum_succ_1y,mcc_sum_sq_succ_30d,mcc_count_succ_1y,mcc_sum_sq_succ_1y,mcc_sum_succ_30d,is_legal_entity,is_mei,is_person
0,2024-02-02 21:43:57.697,275.0,0,DAQQ7TS,322.285714,0.0,411.133179,0.000000,0.0,5048.175078,...,0.0000,0.0,7.523910e+08,1.404402e+11,4232421,1.548410e+12,6.384952e+07,0,0,1
1,2024-02-02 20:49:30.476,2805.5,0,CR4K9VY,66.285714,0.0,35289.128889,170.986536,13.0,334646.251337,...,423018.1025,0.0,7.412321e+08,9.651831e+10,8401389,2.947039e+12,2.877422e+07,0,0,1
2,2024-02-02 15:32:12.033,150.0,0,DSSEHKE,284.142857,0.0,1585.621610,0.000000,0.0,10383.890144,...,0.0000,0.0,2.189377e+09,3.328640e+11,18806623,3.802283e+12,1.733248e+08,0,1,0
3,2024-02-02 13:21:08.696,177.0,0,2C2EP9Y,38.857143,0.0,2659.310168,0.000000,46.0,23495.076815,...,5147.0000,0.0,9.638338e+07,4.847552e+09,2114366,6.598435e+10,8.536190e+06,0,0,1
4,2024-02-02 18:32:29.093,259.0,0,C3NYGCP,39.142857,0.0,1663.235351,0.000000,14.0,13035.453581,...,0.0000,1.0,2.997443e+09,5.486961e+11,31893920,5.762067e+12,2.198331e+08,0,0,1


In [3]:
transactions.shape
# (991.965, 91)

(991965, 91)

## Null

In [4]:
import pandas as pd
from scipy import stats


def analyze_nulls_relationship_with_fraud(df):
    """
    Analyze if null values in each column are related to fraud
    by comparing fraud rates and using chi-square test
    """
    results = []

    # Get total fraud rate for reference
    total_fraud_rate = df['infraction'].mean()

    for column in df.columns:
        if column != 'infraction' and df[column].isnull().any():
            # Calculate fraud rates
            fraud_rate_null = df[df[column].isnull()]['infraction'].mean()
            fraud_rate_not_null = df[df[column].notnull()]['infraction'].mean()

            # Create contingency table for chi-square test
            contingency = pd.crosstab(df[column].isnull(), df['infraction'])

            # Perform chi-square test
            chi2, p_value = stats.chi2_contingency(contingency)[:2]

            # Calculate percentage of nulls
            null_percentage = (df[column].isnull().sum() / len(df)) * 100

            # Calculate relative difference in fraud rates
            if fraud_rate_not_null > 0:
                relative_difference = (fraud_rate_null - fraud_rate_not_null) / fraud_rate_not_null
            else:
                relative_difference = np.inf if fraud_rate_null > 0 else 0

            results.append({
                'Column': column,
                'Null_Percentage': null_percentage,
                'Fraud_Rate_Null': fraud_rate_null,
                'Fraud_Rate_Not_Null': fraud_rate_not_null,
                'Relative_Difference': relative_difference,
                'Chi_Square': chi2,
                'P_Value': p_value,
                'Is_Significant': p_value < 0.05,
                'Conclusion': 'Related to fraud' if p_value < 0.05 else 'Random'
            })

    # Convert to DataFrame and sort by significance
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('P_Value')

    # Format percentages and float values
    results_df['Null_Percentage'] = results_df['Null_Percentage'].round(2)
    results_df['Fraud_Rate_Null'] = results_df['Fraud_Rate_Null'].round(4)
    results_df['Fraud_Rate_Not_Null'] = results_df['Fraud_Rate_Not_Null'].round(4)
    results_df['Relative_Difference'] = results_df['Relative_Difference'].round(2)
    results_df['P_Value'] = results_df['P_Value'].apply(lambda x: f'{x:.2e}')

    return results_df


# Load the data
df = pd.read_csv('raw_transactions.csv')

# Run the analysis
results = analyze_nulls_relationship_with_fraud(df)

# Print results
print("\nNull Values Analysis Results:")
print("\nColumns with significant relationship to fraud (p < 0.05):")
significant = results[results['Is_Significant']]
print(significant[
          ['Column', 'Null_Percentage', 'Fraud_Rate_Null', 'Fraud_Rate_Not_Null', 'Relative_Difference', 'P_Value']])

print("\nColumns with random null distribution (p >= 0.05):")
not_significant = results[~results['Is_Significant']]
print(not_significant[
          ['Column', 'Null_Percentage', 'Fraud_Rate_Null', 'Fraud_Rate_Not_Null', 'Relative_Difference', 'P_Value']])

# Save detailed results
results.to_csv('null_analysis_results.csv', index=False)


Null Values Analysis Results:

Columns with significant relationship to fraud (p < 0.05):
                                               Column  Null_Percentage  \
1                     mc_atm_br_tx_out_count_lifetime             0.38   
3                         mc_billpay_br_count_last_1y             0.39   
5                             mc_pur_br_count_last_1y             0.39   
4               mc_credit_transfer_out_br_sum_last_1y             1.25   
6                        mc_billpay_br_count_last_15d             1.59   
7                      mc_atm_br_tx_out_count_last_1y             0.39   
10           mc_credit_transfer_out_br_count_lifetime             0.38   
9              mc_credit_transfer_in_br_sum_last_180d             1.11   
14             mc_credit_transfer_in_br_sum_last_120d             1.27   
13                           mc_pur_br_count_last_15d             1.59   
11  mc_credit_transfer_out_br_count_sanction_last_30d             0.89   
12           mc_credi

In [5]:
caract_df(transactions)

Nº linhas: 991965
Nº colunas: 91
Nº linhas duplicadas: 0

Nº de vazios*:
	mc_weeks_signup: 11252 - 1.13%
	mc_atm_br_tx_out_count_lifetime: 3799 - 0.38%
	mc_credit_transfer_out_br_sum_last_30d: 27259 - 2.75%
	mc_billpay_br_count_last_1y: 3827 - 0.39%
	mc_credit_transfer_out_br_sum_last_1y: 12432 - 1.25%
	mc_pur_br_count_last_1y: 3827 - 0.39%
	mc_billpay_br_count_last_15d: 15779 - 1.59%
	mc_atm_br_tx_out_count_last_1y: 3827 - 0.39%
	mc_sanction_count_last_2d: 203567 - 20.52%
	mc_credit_transfer_in_br_sum_last_180d: 11013 - 1.11%
	mc_credit_transfer_out_br_count_lifetime: 3799 - 0.38%
	mc_credit_transfer_out_br_count_sanction_last_30d: 8817 - 0.89%
	mc_credit_transfer_out_br_count_last_15d: 15779 - 1.59%
	mc_pur_br_count_last_15d: 15779 - 1.59%
	mc_credit_transfer_in_br_sum_last_120d: 12627 - 1.27%
	mc_pur_br_count_lifetime: 3799 - 0.38%
	mc_credit_transfer_in_br_count_last_30d: 8817 - 0.89%
	mc_credit_transfer_in_br_sum_last_90d: 14011 - 1.41%
	mc_credit_transfer_out_br_count_last_1y: 38

In [6]:



def treat_missing_values(df):
    """
    Comprehensive missing value treatment preserving the signal from null values
    """
    # Create a copy to avoid modifying the original dataframe
    df_treated = df.copy()

    # Store columns to process (excluding target and non-numeric columns)
    columns_to_process = df.select_dtypes(include=['float64', 'int64']).columns
    columns_to_process = [col for col in columns_to_process if
                          not col in ['infraction', 'merchant_id', 'event_created_at']]

    # # Create null indicator features
    # for column in columns_to_process:
    #     if df[column].isnull().any():
    #         # Create missing indicator
    #         df_treated[f'{column}_is_null'] = df[column].isnull().astype(int)

    # Different imputation strategies

    # knn_imputer = KNNImputer(n_neighbors=5)

    # 2. Forward fill for time-dependent features
    df_treated['event_created_at'] = pd.to_datetime(df_treated['event_created_at'])
    df_treated = df_treated.sort_values('event_created_at')

    for column in columns_to_process:
        if df[column].isnull().any():
            # Calculate the fraud rate difference for null vs non-null
            fraud_rate_null = df[df[column].isnull()]['infraction'].mean()
            fraud_rate_non_null = df[df[column].notnull()]['infraction'].mean()
            ratio = fraud_rate_null / fraud_rate_non_null if fraud_rate_non_null > 0 else np.inf

            # Choose imputation strategy based on fraud rate ratio
            if ratio > 10:  # If nulls are much more likely to be fraud
                # Use a special value (e.g., -999) to preserve the signal
                df_treated[column] = df_treated[column].fillna(-999999)
            else:
                df_treated[column] = df_treated[column].fillna(df_treated[column].median())

            # else:
            #    df_treated[column] = pd.DataFrame(
            #         knn_imputer.fit_transform(df_treated[[column]]),
            #         columns=[column],
            #         index=df_treated.index
            #     )

    # # Fill any remaining nulls with median
    # for column in columns_to_process:
    #     if df_treated[column].isnull().any():
    #         df_treated[column] = df_treated[column].fillna(df_treated[column].median())

    return df_treated


# Example usage:
df = pd.read_csv('raw_transactions.csv')

# Get original null counts
null_counts_before = df.isnull().sum()
print("\nNull counts before treatment:")
print(null_counts_before[null_counts_before > 0])

# Apply treatment
df_treated = treat_missing_values(df)

# Get null counts after treatment
null_counts_after = df_treated.isnull().sum()
print("\nNull counts after treatment:")
print(null_counts_after[null_counts_after > 0])

# Print new features created
new_features = [col for col in df_treated.columns if col not in df.columns]
print("\nNew indicator features created:")
print(new_features)


Null counts before treatment:
mc_weeks_signup                                       11252
mc_atm_br_tx_out_count_lifetime                        3799
mc_credit_transfer_out_br_sum_last_30d                27259
mc_billpay_br_count_last_1y                            3827
mc_credit_transfer_out_br_sum_last_1y                 12432
mc_pur_br_count_last_1y                                3827
mc_billpay_br_count_last_15d                          15779
mc_atm_br_tx_out_count_last_1y                         3827
mc_sanction_count_last_2d                            203567
mc_credit_transfer_in_br_sum_last_180d                11013
mc_credit_transfer_out_br_count_lifetime               3799
mc_credit_transfer_out_br_count_sanction_last_30d      8817
mc_credit_transfer_out_br_count_last_15d              15779
mc_pur_br_count_last_15d                              15779
mc_credit_transfer_in_br_sum_last_120d                12627
mc_pur_br_count_lifetime                               3799
mc_credit

In [7]:
df_treated

,event_created_at,amount,infraction,merchant_id,mc_weeks_signup,mc_atm_br_tx_out_count_lifetime,mc_credit_transfer_out_br_sum_last_30d,mc_billpay_br_sum_last_15d,mc_billpay_br_count_last_1y,mc_credit_transfer_out_br_sum_last_1y,...,mc_tx_cp_dbt_sum_sq_fail_last_14d,mc_merchants_ip_count_dist_shared_last_15d,mcc_sum_succ_1y,mcc_sum_sq_succ_30d,mcc_count_succ_1y,mcc_sum_sq_succ_1y,mcc_sum_succ_30d,is_legal_entity,is_mei,is_person
473558,2023-07-01 00:01:02.493914,107.5,0,D62L9GQ,69.714286,0.0,1440.115473,65.510513,19.0,12698.963010,...,0.00,1.0,2.302048e+09,6.907739e+11,20826995,6.583108e+12,2.301642e+08,0,0,1
806288,2023-07-01 00:01:13.949476,138.5,0,ECF6ENP,71.142857,0.0,1733.728697,718.527985,77.0,9497.237914,...,250.75,0.0,2.144144e+09,3.740267e+11,19358946,3.356191e+12,1.778138e+08,0,0,1
636880,2023-07-01 00:01:32.933130,150.0,0,UK3ZG3M,258.285714,2.0,175.442214,0.000000,0.0,625.039834,...,0.00,0.0,2.144144e+09,3.740267e+11,19358946,3.356191e+12,1.778138e+08,0,0,1
511561,2023-07-01 00:01:41.744530,135.0,0,CZYH6X7,48.857143,0.0,2615.888201,0.000000,4.0,10640.710971,...,0.00,0.0,2.144144e+09,3.740267e+11,19358946,3.356191e+12,1.778138e+08,0,0,1
687219,2023-07-01 00:01:56.158400,418.0,0,G9PY39Y,190.142857,0.0,1406.257142,0.000000,0.0,9565.032061,...,1028.00,0.0,3.039484e+09,1.524340e+11,101862055,2.233361e+12,2.345546e+08,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204941,2024-05-01 23:58:40.446000,600.0,0,PQ2QDEX,65.285714,0.0,6376.565975,204.061248,18.0,89037.053401,...,1768900.00,0.0,2.020037e+08,6.221600e+10,1213249,6.899038e+11,1.966357e+07,0,0,1
26153,2024-05-01 23:58:45.363000,200.0,0,4V2CR26,253.142857,0.0,5533.079508,0.000000,61.0,41480.860678,...,0.00,0.0,2.217093e+09,3.983409e+11,18281557,4.095064e+12,1.924288e+08,0,0,1
130562,2024-05-01 23:59:49.887000,145.0,0,CCNGDNG,4.142857,0.0,1217.194475,367.828938,4.0,1217.194475,...,0.00,0.0,2.776819e+09,2.310073e+11,93005969,2.290934e+12,2.214253e+08,0,0,1
211337,2024-05-01 23:59:52.112000,150.0,0,CSHCSKM,185.285714,0.0,591.842026,22.174899,12.0,6399.846418,...,0.00,0.0,1.674791e+09,1.370278e+11,51236362,1.531982e+12,1.394113e+08,0,0,1


In [ ]:

from imblearn.over_sampling import SMOTE
import xgboost as xgb


def evaluate_model(model_name, y_true, y_pred, y_proba):
    """Calculate evaluation metrics with model name prefix"""
    metrics = {
        f'{model_name}_Precision': precision_score(y_true, y_pred, zero_division=0),
        f'{model_name}_Recall': recall_score(y_true, y_pred, zero_division=0),
        f'{model_name}_F1-Score': f1_score(y_true, y_pred, zero_division=0),
        f'{model_name}_Avg Precision': average_precision_score(y_true, y_proba),
    }

    # Calculate Precision@100
    y_proba_series = pd.Series(y_proba, index=y_true.index)
    top_100 = y_proba_series.nlargest(100)
    metrics[f'{model_name}_Precision@100'] = y_true.loc[top_100.index].mean()

    return metrics


def train_and_evaluate(X_train, X_test, y_train, y_test, sampler_name, sampler=None):
    """Train and evaluate both models with given sampler"""
    print(f"\nProcessing {sampler_name}...")
    results = {}

    try:
        # Apply sampling only on training data
        if sampler is not None:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            print(f"Resampled shape: {X_res.shape}, Fraud ratio: {y_res.mean():.4f}")
        else:
            X_res, y_res = X_train, y_train

        class_ratio = len(y_res[y_res == 0]) / len(y_res[y_res == 1])

        # ========== LightGBM ==========
        try:
            lgb_model = lgb.LGBMClassifier(
                n_estimators=1000,
                learning_rate=0.01,
                num_leaves=32,
                scale_pos_weight=class_ratio,
                random_state=42,
                verbose=-1
            )
            lgb_model.fit(X_res, y_res)
            lgb_pred = lgb_model.predict(X_test)
            lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
            results.update(evaluate_model('LGBM', y_test, lgb_pred, lgb_proba))
        except Exception as e:
            print(f"LightGBM Error: {str(e)}")

        # ========== XGBoost ==========
        try:
            xgb_model = xgb.XGBClassifier(
                n_estimators=1000,
                learning_rate=0.01,
                max_depth=5,
                scale_pos_weight=class_ratio,
                eval_metric='aucpr',
                random_state=42,
                use_label_encoder=False
            )
            xgb_model.fit(X_res, y_res)
            xgb_pred = xgb_model.predict(X_test)
            xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
            results.update(evaluate_model('XGB', y_test, xgb_pred, xgb_proba))
        except Exception as e:
            print(f"XGBoost Error: {str(e)}")

        return results

    except Exception as e:
        print(f"General Error: {str(e)}")
        return {k: np.nan for k in [
            'LGBM_Precision', 'LGBM_Recall', 'LGBM_F1-Score', 'LGBM_Avg Precision', 'LGBM_Precision@100',
            'XGB_Precision', 'XGB_Recall', 'XGB_F1-Score', 'XGB_Avg Precision', 'XGB_Precision@100'
        ]}


def main():
    """Main execution flow"""
    # Load and prepare data
    # df = pd.read_parquet('sumup_cas.parquet').sort_values('event_created_at')
    df = df_treated.sort_values('event_created_at').copy()

    # Feature selection
    X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
    y = df['infraction']

    # Time-based split (no shuffle)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2,
        # shuffle=False
    )

    # Define samplers with 20% fraud target
    samplers = {
        'No Sampling': None,
        'Random Under': RandomUnderSampler(sampling_strategy=0.25, random_state=42),
        'SMOTE': SMOTE(sampling_strategy=0.25, random_state=42),
        # 'ADASYN': ADASYN(sampling_strategy=0.25, random_state=42),
        # 'SMOTE-Tomek': SMOTETomek(sampling_strategy=0.25, random_state=42),
        # 'SMOTE-ENN': SMOTEENN(sampling_strategy=0.25, random_state=42)
    }

    # Evaluate all samplers
    results = {}
    for name, sampler in samplers.items():
        results[name] = train_and_evaluate(X_train, X_test, y_train, y_test, name, sampler)

    # Save and display results
    results_df = pd.DataFrame(results).T
    results_df.to_csv('sampling_results.csv', float_format='%.4f')

    print("\nFinal Results:")
    print(results_df)

    return results_df


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

# 1. Load and prepare data
# df = pd.read_parquet('sumup_cas.parquet').sort_values('event_created_at')
df = df_treated.copy()
X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
y = df['infraction']

# Time-based split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

# 2. Define model collection with parameter grids
models = {
    'DecisionTree': (
        DecisionTreeClassifier(),
        {'model__max_depth': [5, 10, 20, 50],
         'model__min_samples_split': [2, 5]}
    ),
    'RandomForest': (
        RandomForestClassifier(),
        {'model__n_estimators': [20, 100, 200, 500],
         'model__max_depth': [5, 10, 20, 50]}
    ),
    'SVM': (
        SVC(probability=True),
        {'model__C': [0.1, 1, 10],
         'model__kernel': ['linear', 'rbf', 'poly']}
    ),
    'XGBoost': (
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        {'model__learning_rate': [0.01, 0.1, 0.25],
         'model__max_depth': [3, 6, 10, 20],
         'model__n_estimators': [20, 50, 100, 200, 500]}
    ),
    'AdaBoost': (
        AdaBoostClassifier(),
        {'model__n_estimators': [20, 50, 100, 200, 500],
         'model__max_depth': [3, 6, 10, 20],
         'model__learning_rate': [0.01, 0.1, 0.2, 0.3]}
    ),
    'LightGBM': (
        lgb.LGBMClassifier(),
        {'model__num_leaves': [16, 32],
         'model__learning_rate': [0.01, 0.1, 0.25]}
    ),
    'MLP': (
        MLPClassifier(early_stopping=True),
        {'model__hidden_layer_sizes': [(50,), (100,), (150,)],
         'model__alpha': [0.0001, 0.001, 0.002]}
    )
}

# 3. Train and validate models
results = {}
best_estimators = {}

for model_name, (estimator, param_grid) in models.items():
    try:
        print(f"\n=== Training {model_name} ===")

        # Create pipeline with RUS and model
        pipeline = Pipeline([
            ('sampler', RandomUnderSampler(sampling_strategy=0.25, random_state=42)),
            ('model', estimator)
        ])

        # Grid search with recall optimization
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring='recall',
            cv=5,
            n_jobs=-1,
            verbose=1
        )

        grid.fit(X_train, y_train)

        # Store best model
        best_estimators[model_name] = grid.best_estimator_

        # Validate on test set
        y_pred = grid.best_estimator_.predict(X_test)
        y_proba = grid.best_estimator_.predict_proba(X_test)[:, 1]

        # Calculate metrics
        results[model_name] = {
            'Best Params': grid.best_params_,
            'Recall': recall_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred),
            'Avg Precision': average_precision_score(y_test, y_proba),
            'Precision@100': pd.Series(y_proba, index=y_test.index)
            .nlargest(100).index.map(y_test).mean()
        }

    except Exception as e:
        print(f"Error with {model_name}: {str(e)}")
        results[model_name] = 'Failed'

# 4. Save and display results
results_df = pd.DataFrame(results).T.sort_values('Recall', ascending=False)
results_df.to_csv('model_selection_results.csv', float_format='%.4f')

print("\nFinal Model Comparison:")
print(results_df[['Recall', 'Precision@100', 'Avg Precision', 'Best Params']])

# 5. Select best model based on validation recall
best_model_name = results_df.index[0]
best_model = best_estimators[best_model_name]
print(f"\nBest Model: {best_model_name} with Recall {results_df.iloc[0].Recall:.4f}")

# Unsupervisioned

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import precision_score, f1_score
from pyod.models.hbos import HBOS
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
import optuna


class UnsupervisedAnomalyDetector:
    def __init__(self, contamination=0.05):
        self.contamination = contamination
        self.models = {}
        self.best_params = {}
        self.scores = {}

    def optimize_isolation_forest(self, X, trial):
        """Optimize IsolationForest hyperparameters"""
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_samples': trial.suggest_float('max_samples', 0.1, 1.0),
            'max_features': trial.suggest_float('max_features', 0.1, 1.0),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'contamination': self.contamination,
            'random_state': 42
        }

        model = IsolationForest(**params)
        model.fit(X)
        scores = -model.score_samples(X)
        return self._calculate_objective(scores)

    def optimize_lof(self, X, trial):
        """Optimize LocalOutlierFactor hyperparameters"""
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 5, 50),
            'leaf_size': trial.suggest_int('leaf_size', 10, 100),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan']),
            'contamination': self.contamination
        }

        model = LocalOutlierFactor(**params)
        scores = -model.fit_predict(X)
        return self._calculate_objective(scores)

    def optimize_hbos(self, X, trial):
        """Optimize HBOS hyperparameters"""
        params = {
            'n_bins': trial.suggest_int('n_bins', 5, 50),
            'alpha': trial.suggest_float('alpha', 0.1, 1.0),
            'tol': trial.suggest_float('tol', 0.1, 0.5),
            'contamination': self.contamination
        }

        model = HBOS(**params)
        model.fit(X)
        scores = model.decision_function(X)
        return self._calculate_objective(scores)

    def _calculate_objective(self, scores):
        """Calculate objective value for optimization"""
        threshold = np.percentile(scores, (1 - self.contamination) * 100)
        predictions = (scores > threshold).astype(int)
        return np.mean(predictions == (scores > np.median(scores)))

    def optimize_models(self, X, n_trials=100):
        """Optimize all models using Optuna"""
        optimization_funcs = {
            'IsolationForest': self.optimize_isolation_forest,
            'LOF': self.optimize_lof,
            'HBOS': self.optimize_hbos
        }

        for name, optimize_func in optimization_funcs.items():
            print(f"\nOptimizing {name}...")
            study = optuna.create_study(direction='maximize')
            study.optimize(lambda trial: optimize_func(X, trial), n_trials=n_trials)
            self.best_params[name] = study.best_params
            print(f"Best parameters for {name}: {study.best_params}")

    def fit_predict(self, X, y=None):
        """Fit and predict with all models"""
        # Initialize models with best parameters
        self.models['IsolationForest'] = IsolationForest(
            **self.best_params['IsolationForest'],
            contamination=self.contamination
        )

        self.models['LOF'] = LocalOutlierFactor(
            **self.best_params['LOF'],
            contamination=self.contamination
        )

        self.models['HBOS'] = HBOS(
            **self.best_params['HBOS'],
            contamination=self.contamination
        )

        # Add other state-of-the-art models with default parameters
        self.models['COPOD'] = COPOD(contamination=self.contamination)
        self.models['ECOD'] = ECOD(contamination=self.contamination)
        self.models['EllipticEnvelope'] = EllipticEnvelope(
            contamination=self.contamination,
            random_state=42
        )

        # Fit and predict with each model
        predictions = {}
        scores = {}

        for name, model in self.models.items():
            print(f"\nFitting {name}...")
            if isinstance(model, LocalOutlierFactor):
                predictions[name] = model.fit_predict(X)
                scores[name] = model.negative_outlier_factor_
            else:
                model.fit(X)
                predictions[name] = model.predict(X)
                scores[name] = model.decision_function(X) if hasattr(model, 'decision_function') \
                    else model.score_samples(X)

        return predictions, scores

    def evaluate(self, predictions, y_true):
        """Evaluate model performance if true labels are available"""
        results = {}

        for name, y_pred in predictions.items():
            # Convert predictions to binary (1 for anomaly, 0 for normal)
            y_pred_binary = (y_pred == -1).astype(int)

            results[name] = {
                'precision': precision_score(y_true, y_pred_binary),
                'recall': recall_score(y_true, y_pred_binary),
                'f1': f1_score(y_true, y_pred_binary)
            }

        return pd.DataFrame(results).T


def main():
    # Load data
    print("Loading data...")
    # df = pd.read_csv('raw_transactions.csv')
    df = df_treated.copy()

    # Prepare features
    X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
    y = df['infraction']  # Only used for evaluation

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Initialize detector
    detector = UnsupervisedAnomalyDetector(contamination=0.05)  # 5% anomalies

    # Optimize models
    detector.optimize_models(X_scaled)

    # Fit and predict
    predictions, scores = detector.fit_predict(X_scaled)

    # Evaluate results
    results = detector.evaluate(predictions, y)
    print("\nModel Performance:")
    print(results)

    # Save results
    results.to_csv('unsupervised_model_results.csv')

    # Create DataFrame with all scores and predictions
    results_df = pd.DataFrame(index=df.index)

    for name in detector.models.keys():
        results_df[f'{name}_score'] = scores[name]
        results_df[f'{name}_prediction'] = predictions[name]

    # Add ensemble score (average of all normalized scores)
    normalized_scores = pd.DataFrame(scores).apply(lambda x: (x - x.mean()) / x.std())
    results_df['ensemble_score'] = normalized_scores.mean(axis=1)

    # Add original data and true labels
    results_df = pd.concat([df, results_df], axis=1)

    # Save detailed results
    results_df.to_csv('unsupervised_detailed_results.csv')

    return detector, results, results_df


if __name__ == "__main__":
    detector, results, results_df = main()

In [ ]:
import numpy as np
import pandas as pd

# For model selection
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import recall_score, classification_report

# Unsupervised anomaly detection models
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# =============================================================================
# Define a custom scoring function for unsupervised models.
# Since these models output -1 for anomalies and 1 for inliers,
# we convert predictions so that:
#    -1 becomes 1 (suspicious) and 1 becomes 0 (normal),
# and then compute recall (i.e. fraction of true frauds that are flagged).
# =============================================================================
from sklearn.metrics import make_scorer


def unsupervised_recall_scorer(estimator, X, y):
    # For LOF and One-Class SVM, ensure that the model has been fitted with novelty=True if needed.
    y_pred = estimator.predict(X)
    # Convert: anomalies (-1) -> 1, inliers (1) -> 0
    y_pred_bin = np.where(y_pred == -1, 1, 0)
    return recall_score(y, y_pred_bin, zero_division=0)


unsupervised_recall = make_scorer(unsupervised_recall_scorer)

# =============================================================================
# Load data (raw, without scaling)
# =============================================================================
print("Loading data...")
# df = pd.read_csv('raw_transactions.csv')
df = df_treated.copy()

# Prepare features by dropping columns not used for prediction.
X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
y = df['infraction']

print(f"Data shape: {X.shape}")
print(f"Number of fraud cases: {sum(y == 1)}")
print(f"Fraud ratio: {sum(y == 1) / len(y):.4%}")

# Train-test split (stratify so that test set maintains the original distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# =============================================================================
# Define unsupervised candidate models with parameter grids.
# Note: For LOF, we must set novelty=True so that it can be used on new (test) data.
# =============================================================================
unsupervised_models = {
    "Isolation Forest": {
        "estimator": IsolationForest(random_state=42),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_samples": [0.5, 0.75, 1.0],
            "contamination": [0.0005, 0.001, 0.005]
        }
    },
    "Local Outlier Factor": {
        # Set novelty=True to enable predicting on unseen data.
        "estimator": LocalOutlierFactor(novelty=True),
        "param_grid": {
            "n_neighbors": [20, 30, 50],
            "contamination": [0.0005, 0.001, 0.005]
        }
    },
    "One-Class SVM": {
        "estimator": OneClassSVM(kernel='rbf'),
        "param_grid": {
            "nu": [0.0005, 0.001, 0.005],  # nu is an upper bound on the fraction of training errors
            "gamma": ['scale', 0.001, 0.01, 0.1]
        }
    }
}

# =============================================================================
# Hyperparameter tuning for each unsupervised model using GridSearchCV.
# We use our custom unsupervised_recall scorer.
# =============================================================================
best_unsupervised_models = {}

print("\n--- Hyperparameter Tuning for Unsupervised Models ---")
for model_name, model_info in unsupervised_models.items():
    print(f"\n----- {model_name} -----")
    grid = GridSearchCV(
        estimator=model_info["estimator"],
        param_grid=model_info["param_grid"],
        scoring=unsupervised_recall,
        cv=5,  # 5-fold cross-validation on the training set
        n_jobs=-1,
        verbose=1
    )
    # Fit grid search on the training set (which remains unaltered)
    grid.fit(X_train, y_train)
    print(f"Best parameters for {model_name}: {grid.best_params_}")
    print(f"Best CV Recall for {model_name}: {grid.best_score_:.4f}")
    best_unsupervised_models[model_name] = grid.best_estimator_

# =============================================================================
# Evaluate each candidate unsupervised model on the test set.
# =============================================================================
print("\n--- Evaluation on Test Set for Unsupervised Models ---")
unsupervised_results = {}

for model_name, model in best_unsupervised_models.items():
    print(f"\nEvaluating {model_name}...")
    # Use predict on test set; convert -1 to 1 (suspicious) and 1 to 0 (normal)
    y_pred = model.predict(X_test)
    y_pred_bin = np.where(y_pred == -1, 1, 0)

    recall_val = recall_score(y_test, y_pred_bin, zero_division=0)
    print(classification_report(y_test, y_pred_bin, zero_division=0))

    unsupervised_results[model_name] = recall_val
    print(f"Test Recall for {model_name}: {recall_val:.4f}")

# Select the best unsupervised model (based on recall on test set)
best_unsupervised_model_name = max(unsupervised_results, key=unsupervised_results.get)
print(
    f"\nSelected Unsupervised Model: {best_unsupervised_model_name} with Test Recall = {unsupervised_results[best_unsupervised_model_name]:.4f}")

# =============================================================================
# With the winning unsupervised model, generate anomaly scores and rank transactions.
# =============================================================================
winning_model = best_unsupervised_models[best_unsupervised_model_name]

# For models that support decision_function (e.g., Isolation Forest, One-Class SVM, LOF with novelty=True)
# lower scores (more negative) indicate higher anomaly.
if hasattr(winning_model, "decision_function"):
    anomaly_scores = winning_model.decision_function(X)
else:
    # If decision_function is not available, we can use negative of predict_proba if available,
    # or simply use the raw predictions.
    anomaly_scores = -winning_model.predict(X)

# Append anomaly scores to the original dataframe (use the full dataset for ranking)
df['anomaly_score'] = anomaly_scores

# Rank transactions: most suspicious first (i.e. smallest decision function score)
df_ranked = df.sort_values('anomaly_score', ascending=True)

# For instance, select the top 100 most suspicious transactions:
top_100_anomalies = df_ranked.head(100)
print("\nTop 100 most suspicious transactions:")
print(top_100_anomalies[['anomaly_score', 'infraction']])

# The ranked list can now be used by your fraud team to prioritize cases for human evaluation.
